In [229]:
import sys
# !{sys.executable} -m pip install opentrons
# !pip install opentrons

In [230]:
## Imports
from opentrons import protocol_api, execute, simulate
from opentrons.types import Location, Point
import pandas as pd
import numpy as np

#from opentrons import containers, instruments


# Call protocol to simulate or Execute

In [231]:
# protocol = simulate.get_protocol_api('2.10')
protocol = execute.get_protocol_api('2.19')

#Arthur added this
#protocol = execute.get_protocol_api("2.19")
#protocol.home()


# Define labwear

In [232]:
## pipette and labware
tiprack = protocol.load_labware('opentrons_96_tiprack_300ul', '6')
pip = protocol.load_instrument('p300_single_gen2', 'right', tip_racks=[tiprack])
stock = protocol.load_labware('corning_6_wellplate_16.8ml_flat',location='3')
stock2 = protocol.load_labware('corning_6_wellplate_16.8ml_flat',location='2')
plate = protocol.load_labware('opentrons_96_wellplate_200ul_pcr_full_skirt','1')
# plate.set_offset(x=0, y=0, z=.5)
plate.set_offset(x=0, y=0, z=33)

#plate = containers.load('96-flat', 'B1')

# Import container
# tiprack = containers.load('tiprack-200ul', 'A1')
# trash = containers.load('point', 'D2')
# #from data import data_colordf
# trash = containers.load('point', 'D2')
#flat_plate = protocol.load_labware('nest_96_wellplate_200ul_flat','2')

# Define locations of stock solution

In [233]:
start_stock = {
    'magenta_loc': [stock.wells()[0].center(),stock.wells()[1].center()],#stock.wells()[0].center()
    #'water_loc': loc_stock(3,stock.wells()[0].center())+loc_stock(3,stock.wells()[1].center()),#stock.wells()[0].center()
    'magenta_vol':[20000,20000] ,# adjust the volumes for each(6) concentration
    'cyan_loc':[stock.wells()[2].center(),stock.wells()[3].center()], #stock.wells()[1].center()
    #'dmf_loc':loc_stock(3,stock.wells()[2].center())+loc_stock(3,stock.wells()[3].center()), #stock.wells()[1].center()
    'cyan_vol':[20000,20000],# adjust the volumes for each(6) concentration
    'yellow_loc':[stock.wells()[4].center(),stock.wells()[5].center()],#stock.wells()[2].center()
    #'ethanol_loc':loc_stock(3,stock.wells()[4].center())+loc_stock(3,stock.wells()[5].center()),#stock.wells()[2].center()
    'yellow_vol':[20000,20000],# adjust the volumes for each(6) concentration
    'black_loc':[stock2.wells()[0].center(),stock2.wells()[1].center()],#stock.wells()[2].center()
    #'ethanol_loc':loc_stock(3,stock.wells()[4].center())+loc_stock(3,stock.wells()[5].center()),#stock.wells()[2].center()
    'black_vol':[20000,20000]# adjust the volumes for each(6) concentration
}
start_stock = pd.DataFrame(data=start_stock)


# Relevant Functions

In [234]:
# def loc_stock(n_samples,fill):
#     array = []
#     #fill array
#     for i in range(n_samples): 
#         array.append(fill) 
#     return array

# def loc_mixing(start,n_sample,plate): 
#     array = []
#     for i in range(n_sample):
#         array.append(plate.wells()[i+start].center())
#     return array

In [235]:
# function for moving liquid since repetitive
def pipette_action(protocol, pipette, stock_volume_to_pull, stock_position_to_pull, destination_well,stockname):
    pipette.aspirate(stock_volume_to_pull, stock_position_to_pull)
    pipette.dispense(stock_volume_to_pull, destination_well)
    pipette.touch_tip()  
    #pipette.blow_out(destination_well)

In [236]:
def stock_in_all(stock,stock_location,tip_max_vol,dis_tray,name):
    '''
    Args- 
    stock: 1D array of all the stock volumes
    stock_location: 1D array of the stock locations corresponding to volumes
    tip_max_vol: float num of the max volume tip can hold
    dis_tray: 1D array of all the locaitons for dispensing
    name: stock name str

    Function-
    Determine the number of times needed to get stock and dispence. 
    If the volume is larger than the tip can hold, automatically determined how many times it has to aspirate and dispence. 
    Care since stock, stock_location, dis_tray are all arrays
    '''
#     i = 0
    for i in range(len(stock)):
        stock_amount = stock[i]
        if stock_amount == 0:
            pass
        elif stock[i] >= tip_max_vol: # eg:300 is max tip volume
            n = stock[i]//tip_max_vol
            if n > 0: 
                for count in range(int(n)):
                    pipette_action(protocol, 
                            pipette = pip, 
                            stock_volume_to_pull = tip_max_vol, 
                            stock_position_to_pull = stock_location[i], 
                            destination_well = dis_tray[i],
                            stockname= name )
                    stock_amount -= tip_max_vol
                    
                    if stock_amount <= stock[i] - tip_max_vol*n and stock_amount != 0:
                        pipette_action(protocol, 
                                    pipette = pip, 
                                    stock_volume_to_pull = (stock[i] - tip_max_vol*n),
                                    stock_position_to_pull = stock_location[i], 
                                    destination_well = dis_tray[i],
                                    stockname= name )
        else: 
            pipette_action(protocol, 
                           pipette = pip, 
                           stock_volume_to_pull = stock[i], 
                           stock_position_to_pull = stock_location[i], 
                           destination_well = dis_tray[i],
                           stockname= name )
        

In [237]:
def loc_stock(n_samples,fill):
    array = []
    #fill array
    for i in range(n_samples): 
        array.append(fill) 
    return array

def loc_mixing(start,n_sample): 
    array = []
    for i in range(n_sample):
        array.append(plate.wells()[i+start].top(z = 85)) # Change elevation here using top(z=90)
#         array.append(plate.wells()[i+start].center())
    return array

#let = 'D'
#RGB_stocksandlocations = {'mix_loc': [plate[str(let)+'1'].center(), plate[str(let)+'2'].center(),plate[str(let)+'3'].center(),plate[str(let)+'4'].center(),plate[str(let)+'5'].center(),plate[str(let)+'6'].center(),plate[str(let)+'7'].center(),plate[str(let)+'8'].center(),plate[str(let)+'9'].center(),plate[str(let)+'10'].center(),plate[str(let)+'11'].center(),plate[str(let)+'12'].center()], 

In [238]:
##Calling the function
# N_samples = 96
N_samples = 8

loc_red_stoc = loc_stock(N_samples,start_stock['magenta_loc'][0])
loc_blue_stoc = loc_stock(N_samples,start_stock['cyan_loc'][0])
loc_yellow_stoc = loc_stock(N_samples,start_stock['yellow_loc'][0])
loc_black_stoc = loc_stock(N_samples,start_stock['black_loc'][0])

loc_mixing = loc_mixing(0, N_samples) #Start is 0 - 95, A: 0-11, B: 12-23, 
loc_mixing

[Location(point=Point(x=14.38, y=74.24, z=134.0), labware=A1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on slot 1, is_meniscus=False),
 Location(point=Point(x=14.38, y=65.24, z=134.0), labware=B1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on slot 1, is_meniscus=False),
 Location(point=Point(x=14.38, y=56.24, z=134.0), labware=C1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on slot 1, is_meniscus=False),
 Location(point=Point(x=14.38, y=47.24, z=134.0), labware=D1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on slot 1, is_meniscus=False),
 Location(point=Point(x=14.38, y=38.24, z=134.0), labware=E1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on slot 1, is_meniscus=False),
 Location(point=Point(x=14.38, y=29.24, z=134.0), labware=F1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on slot 1, is_meniscus=False),
 Location(point=Point(x=14.38, y=20.24, z=134.0), labware=G1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on

# Volumes and destination locations

In [239]:
# # Required libraries
# import numpy as np
# import pandas as pd
# from pyDOE import lhs


# # Function to generate LHS samples satisfying a sum constraint
# def latin_hypercube_sum_constraint(n_samples, n_variables, sum_constraint, seed=None):
#     # Generate Latin Hypercube Samples
#     if seed is not None:
#         np.random.seed(seed)
#     lhs_samples = lhs(n_variables, samples=n_samples, criterion='maximin')

#     # Transform samples to satisfy the sum constraint
#     for i in range(n_samples):
#         current_sum = np.sum(lhs_samples[i, :])
#         lhs_samples[i, :] *= sum_constraint / current_sum

#     return lhs_samples

# # Parameters
# n_samples = 96  # Total number of samples
# n_variables = 3  # Number of color components (R, G, B)
# sum_constraint = 270.0  # Sum constraint for each sample
# seed = 42  # Seed for reproducibility

# # Generate LHS samples
# lhs_samples = latin_hypercube_sum_constraint(
#     n_samples=n_samples, 
#     n_variables=n_variables, 
#     sum_constraint=sum_constraint, 
#     seed=seed
# )

# # Convert to DataFrame and split into respective color components
# df_lhs = pd.DataFrame(lhs_samples, columns=['Red', 'Blue', 'Yellow'])

In [240]:
# RGB_stocksandlocations = {
#     'mix_loc': loc_mixing,
# 'stock_red_vol': df_lhs['Red'].tolist(),
# 'stock_blue_vol': df_lhs['Blue'].tolist(),
# 'stock_yellow_vol': df_lhs['Yellow'].tolist()
#         }
# RGB_stocks = pd.DataFrame(data=RGB_stocksandlocations)
# RGB_stocks.head()
  

test (4 samples)

In [241]:
# RGB_stocksandlocations = {
#     'mix_loc': loc_mixing,
    
# 'stock_red_vol': [55.912295139203444, 29.315613295486525, 151.3898373145717, 85.17266010030099] ,
# 'stock_blue_vol':[71.05809443170705, 83.86700699218619, 4.506169411300044, 48.14905083668657],
# 'stock_yellow_vol':[56.95880807134397, 114.66529822858064, 89.51441943891615, 5.309367365920778] ,
# 'stock_black_vol':[86.07080235774558, 42.15208148374661, 24.58957383521207, 131.36892169709168] ,
#         }
# RGB_stocks = pd.DataFrame(data=RGB_stocksandlocations)
# RGB_stocks.head()

In [242]:
# RGB_stocksandlocations = {
#     'mix_loc': loc_mixing,
    
# 'stock_red_vol': [86.36187845023132, 137.833831936324, 197.98300457990877, 29.220294620388053] ,
# 'stock_blue_vol':[47.95095946255746, 88.13205018746672, 71.99724904671957, 143.55372962677262],
# 'stock_yellow_vol':[135.68716208721125, 44.03411787620927, 0.01974637337165445, 97.22597575283929] ,
#         }
# RGB_stocks = pd.DataFrame(data=RGB_stocksandlocations)
# RGB_stocks.head()

In [243]:
# #test purple

# RGB_stocksandlocations = {
#     'mix_loc': loc_mixing,
    
# 'stock_red_vol': [113.96, 113.96, 113.96, 113.96] ,
# 'stock_blue_vol':[70.13, 70.13, 70.13, 70.13],
# 'stock_yellow_vol':[0, 0, 0, 0] ,
# 'stock_black_vol':[85.9, 85.9, 85.9, 85.9] ,
#         }
# RGB_stocks = pd.DataFrame(data=RGB_stocksandlocations)
# RGB_stocks.head()

8 samples (1st sampling)

In [244]:
import pandas as pd

# seed 1

# Define the dictionary
RGB_stocksandlocations = {
    'mix_loc': loc_mixing,  # Note: loc_mixing needs to be handled separately
    'stock_red_vol': [77.44581101551692, 11.546358436565134, 78.69476237106129, 133.92480929691936, 23.45725442511232, 34.214508151236785, 80.65295720413329, 102.90089286630496
                     ],
    'stock_blue_vol': [93.01478492294865, 148.52689122118792, 46.77838891842151, 30.08010518248808, 6.222927761134619, 79.06778736672267, 48.64187592001704, 106.36728416140619
                      ],
    'stock_yellow_vol': [37.24451914443951, 49.84456756420468, 72.45247741260627, 2.116495407929885, 104.2715869005695, 89.11625704033183, 86.33764924205805, 55.55365548974068
                        ],
    'stock_black_vol': [62.29488491709496, 60.08218277804225, 72.07437129791093, 103.87859011266265, 136.04823091318354, 67.60144744170866, 54.36751763379163, 5.178167482548181
                       ]
}

# Verify lengths of all columns
print("Length of mix_loc:", len(loc_mixing))  # Replace 'loc_mixing' with actual data or a placeholder
print("Length of stock_red_vol:", len(RGB_stocksandlocations['stock_red_vol']))
print("Length of stock_blue_vol:", len(RGB_stocksandlocations['stock_blue_vol']))
print("Length of stock_yellow_vol:", len(RGB_stocksandlocations['stock_yellow_vol']))


# Create a DataFrame
RGB_stocks = pd.DataFrame(RGB_stocksandlocations)

# Show the first few rows
RGB_stocks.head()

Length of mix_loc: 8
Length of stock_red_vol: 8
Length of stock_blue_vol: 8
Length of stock_yellow_vol: 8


,mix_loc,stock_red_vol,stock_blue_vol,stock_yellow_vol,stock_black_vol
0,"Location(point=Point(x=14.38, y=74.24, z=134.0...",77.445811,93.014785,37.244519,62.294885
1,"Location(point=Point(x=14.38, y=65.24, z=134.0...",11.546358,148.526891,49.844568,60.082183
2,"Location(point=Point(x=14.38, y=56.24, z=134.0...",78.694762,46.778389,72.452477,72.074371
3,"Location(point=Point(x=14.38, y=47.24, z=134.0...",133.924809,30.080105,2.116495,103.878590
4,"Location(point=Point(x=14.38, y=38.24, z=134.0...",23.457254,6.222928,104.271587,136.048231


1st iteration

In [245]:
# import pandas as pd

# # seed 1

# # Define the dictionary
# RGB_stocksandlocations = {
#     'mix_loc': loc_mixing,  # Note: loc_mixing needs to be handled separately
#     'stock_red_vol': [12.406199934125254, 95.81498500085661, 89.46574902543512, 128.9499634073578, 126.7005142916312, 42.25885046415203, 214.95512774660503, 149.04506567750613, 120.25900211428964, 111.58874343137764, 105.83540365225632, 94.50745629129638, 194.29230979500113, 210.446097509733, 20.313419748661087, 137.7681379771849, 192.5281233990067, 101.98850144645694, 98.80413010392428, 7.747840074570097, 11.14993794783053, 108.92479392240936, 47.46661103130435, 137.7319503069057, 117.22337426672267, 33.98914319273048, 97.29268241669196, 76.98369272870765, 110.97146168544192, 56.63498022227643, 171.03059099500703, 104.36975879949773, 98.81800501591796, 99.14919337808858, 173.3859557574867, 61.767463042461195, 156.96672821585983, 225.52849949448995, 197.38859535376037, 173.0970289779433, 105.91934542384013, 180.3876326230617, 116.83033171370596, 66.52426830713506, 117.30065126996092, 104.03551903616268, 149.4089300264498, 33.286135620369485, 89.00786627775345, 99.4136174005049, 176.27159886081444, 118.93215980447125, 129.13347407591687, 97.5492776305249, 73.39118850159048, 47.13954499389328, 92.19217219622254, 114.78280122057245, 86.54804432143965, 104.82314682243651, 65.01604477286716, 172.49523376324169, 123.80270547714134, 57.47976552950964, 47.8382910994491, 43.49511172805596, 128.4932463121554, 113.24939666150149, 117.84147639471404, 151.39203115881233, 113.38329032819216, 120.60655978179032, 47.301348016573044, 134.22670480869772, 100.64970530127137, 194.5284907275686, 101.60907947387508, 185.24486689932672, 127.00437164041936, 169.08476063230387, 92.96427390421475, 6.991795500983898, 190.4167070447581, 119.98564011668687, 158.83446820327592, 121.27603855760232, 112.66824634645785, 156.28128319478964, 116.13485500198149, 193.20911128180867, 150.60954113627866, 149.42675392819413, 173.51956993230553, 156.5728858177899, 90.9039563741101, 59.50754865649246
# ],
#     'stock_blue_vol': [99.85742885000515, 58.359036997558256, 48.08916424159009, 16.17315128819646, 104.78235718417751, 156.5297231202577, 21.92344559764362, 44.92535221095266, 76.30678443138423, 73.08209903307652, 97.71889064972534, 60.00346026155635, 52.223760949735365, 43.36849956275356, 35.7916322035725, 87.68272878487171, 51.523945725506294, 108.73870009655415, 31.749295897129095, 108.02854652896046, 121.70232764877431, 62.45937959137377, 195.99422621619, 74.03241153411572, 54.61041557450546, 94.48602958926614, 68.4424291064621, 63.00970755147926, 41.270411793686804, 95.65295093151157, 64.1180543805703, 20.156668415204766, 74.54262875316432, 35.71170519886891, 70.94918570205844, 117.78717934582626, 85.33768942180173, 35.16933774382813, 57.208752148064505, 9.120685582144663, 102.73923620258999, 55.70159768782896, 53.502610873304725, 8.2166015867707, 56.17860220657613, 34.3238132592026, 34.923080960978744, 5.34085135486697, 99.55922703844854, 77.25499539941244, 83.5568800077511, 42.21886757064756, 56.21285494722325, 17.69844129021209, 79.74845256458913, 91.34441779224957, 17.75495767096371, 67.22363507641921, 59.9317690500709, 73.15666925362842, 66.45586868403872, 30.940310801150115, 61.42495232874044, 100.99379337356486, 50.777623024129554, 152.91136227208096, 15.139682088858454, 74.07957662798243, 19.86250210973088, 23.723427626298342, 51.34556384994348, 76.75870955702719, 48.529969255014905, 73.00437277077316, 44.419487231296195, 7.278252360852216, 123.44845448424863, 25.28231651277215, 87.99429233481234, 28.679164365312435, 7.8169547595024635, 126.60110661420184, 45.09282777407884, 11.32559925828756, 33.752089507125085, 88.75165402642705, 57.66848128386318, 47.294617175002024, 60.932395674376416, 0.46648032631353215, 34.96738423863075, 28.243468617416003, 56.58255190788991, 82.37089651895619, 8.85464478218057, 42.22578355680871
# ],
#     'stock_yellow_vol': [31.710974213251426, 7.156522614098014, 25.108795768301945, 5.583205634282199, 26.164707823956103, 33.409983293601876, 9.815318867108875, 24.02856749628718, 15.164102129615237, 3.764594050138054, 22.699880435274423, 11.63931905632586, 13.366071094070387, 12.114957044238464, 36.25010535055168, 21.112371252676876, 15.79546786178541, 14.041330813788575, 25.617337384479622, 57.220486823488635, 34.839358924166646, 11.105545852375561, 15.895995487064482, 5.420594601059732, 3.6423097907613573, 26.581291627775123, 4.7675150620813564, 7.070948633875963, 15.069434770385644, 22.845937575852798, 7.245844278033904, 0.4706194403724898, 6.061527388520394, 19.730499563377535, 1.0334094534272569, 40.1313117694772, 23.15151652335578, 3.37889080483469, 2.107192519723499, 35.301643543109726, 18.300170834436123, 10.216910912005945, 16.082811905943625, 21.96209182603796, 9.149826765501757, 19.496465798668666, 9.435946459196611, 149.7574232581739, 7.6087701263296355, 19.26683671582036, 9.188825343945469, 22.309897269128157, 15.820107392425516, 7.0742966025620335, 14.429460215176103, 20.178775849183403, 45.116684246151294, 13.710109602927563, 28.588315568962333, 7.61078505599276, 34.36188277422738, 29.449343635901613, 2.3503010210893636, 29.549881334882258, 14.477318666058526, 42.953265575826535, 19.225680058724492, 2.5089654304486544, 16.818105402721596, 18.364654991888234, 16.308123228939813, 1.6674253022268835, 30.911425863831003, 17.08548123095619, 0.41839536717129117, 6.029912106005547, 24.0537842754064, 22.892782140772738, 11.528308267571974, 14.846331137459906, 42.7027696120436, 22.61593145536719, 13.485587641619158, 2.0286743352537338, 17.105783527691596, 19.366555140156517, 6.5655955557755625, 16.13090298609366, 13.884722095623236, 44.395005034819356, 24.390549568971746, 0.7629240575380316, 23.46412075805705, 8.617208297291308, 25.32764779542101, 10.662586761876634
#                         ],
#     'stock_black_vol': [126.02539700261818, 108.66945538748715, 107.33629096467287, 119.29367967016353, 12.352420700235191, 37.80144312198836, 23.306107788642468, 52.001014615254064, 58.27011132471088, 81.56456348540777, 43.74582526274396, 103.8497643908214, 10.11785816119313, 4.070445883274985, 177.6448426972147, 23.436761985266536, 10.152463013701617, 45.23146764320032, 113.82923661446698, 97.0031265729808, 102.30837547922854, 87.51028063384132, 10.643167265441187, 52.815043557918884, 94.52390036801049, 114.94353559022827, 99.49737341476458, 122.93565108593712, 102.6886917504856, 94.86613127035922, 27.605510346388787, 145.00295334492503, 90.57783884239731, 115.40860185966498, 24.631449087027605, 50.31404584223535, 4.54406583898268, 5.923271956847224, 13.295459978451635, 52.480641896802325, 43.041247539133735, 23.69385877710339, 83.5842455070457, 173.29703828005626, 87.37091975796118, 112.14420190596606, 76.23204255337482, 81.61558976658966, 73.82413655746839, 74.06455048426228, 0.9826957874890135, 86.53907535575306, 68.83356358443437, 147.67798447670094, 102.4308987186443, 111.33726136467374, 114.93618588666243, 74.2834541000808, 94.9318710595271, 84.40939886794231, 104.16620376886677, 37.11511179970661, 82.42204117302884, 81.97655976204322, 156.90676721036283, 30.64026042403653, 107.14139154026164, 80.1620612800674, 115.47791609283348, 76.51988622300107, 88.96302259292457, 70.96730535895561, 143.25725686458105, 45.68344118957292, 124.51241210026114, 62.16334480557364, 20.88868176646995, 36.58003444712839, 43.47302775719633, 57.38974386492378, 126.51600172423916, 113.79116642944707, 21.004877539543877, 136.66008628977178, 60.307658761907405, 40.60575227581411, 93.09767681390342, 50.29319664411467, 79.04802722801882, 31.929403357058472, 60.03252505611883, 91.56685339685187, 16.433757401747535, 22.439009365962573, 144.91375104828833, 157.6040810248222
#                        ]
# }

# # Verify lengths of all columns
# print("Length of mix_loc:", len(loc_mixing))  # Replace 'loc_mixing' with actual data or a placeholder
# print("Length of stock_red_vol:", len(RGB_stocksandlocations['stock_red_vol']))
# print("Length of stock_blue_vol:", len(RGB_stocksandlocations['stock_blue_vol']))
# print("Length of stock_yellow_vol:", len(RGB_stocksandlocations['stock_yellow_vol']))


# # Create a DataFrame
# RGB_stocks = pd.DataFrame(RGB_stocksandlocations)

# # Show the first few rows
# RGB_stocks.head()

In [246]:
red_stock_vol    = pd.Series.to_list(RGB_stocks['stock_red_vol'])
blue_stock_vol   = pd.Series.to_list(RGB_stocks['stock_blue_vol'])
yellow_stock_vol = pd.Series.to_list(RGB_stocks['stock_yellow_vol'])
black_stock_vol = pd.Series.to_list(RGB_stocks['stock_black_vol'])

red_stock_loc   = start_stock['magenta_loc'][0]
blue_stock_loc    = start_stock['cyan_loc'][0]
yellow_stock_loc = start_stock['yellow_loc'][0]
black_stock_loc = start_stock['black_loc'][0]

dispense = pd.Series.to_list(RGB_stocks['mix_loc'])

# Run the protocol

In [ ]:
## commands
protocol.home()
pip.transfer(red_stock_vol,red_stock_loc ,dispense,blow_out=True,blowout_location="destination well" ,new_tip='once',trash = False)
# pip.drop_tip()
# move blue 
pip.transfer(blue_stock_vol,blue_stock_loc ,dispense,blow_out=True,blowout_location="destination well" ,new_tip='once',trash = False) 
# pip.drop_tip()
# # move yellow
pip.transfer(yellow_stock_vol,yellow_stock_loc,dispense,blow_out=True,blowout_location="destination well",new_tip='once',trash = False) 
# pip.drop_tip()
# move black
pip.transfer(black_stock_vol,black_stock_loc,dispense,blow_out=True,blowout_location="destination well",new_tip='once',trash = False) 
protocol.home()

for line in protocol.commands(): 
    print(line)

#for i in range(1,len(dispense)+1):
    #pip.mix(3, 100, plate[let+str(i)])   # mix 3 times, 100uL, in all plates 


In [ ]:
# protocol.home()

In [ ]:
# pip.drop_tip()